# TinyGPT2 C4 Fit Reproduction Analysis

This notebook analyzes the pulled summary artifacts for
`gpt2_tiny_c4_ppt_reproduction`.

Goals:
1. Compare the analytic fit blocks against the original pre-pretraining seed-derived subspace.
2. Plot downstream pretraining loss curves by initialization, using training tokens on the x-axis.


In [1]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

%load_ext autoreload
%autoreload 2


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / ".git").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
sys.path.insert(0, str(REPO_ROOT / "src" / "platonic_init"))
sys.path.insert(0, str(REPO_ROOT / "src"))

import aesthetics as aes  # noqa: E402

sns = aes.sns

EXPERIMENT_NAME = "gpt2_tiny_c4_ppt_reproduction"
EXPERIMENT_ROOT = REPO_ROOT / "artifacts" / "experiments" / EXPERIMENT_NAME
ANALYSIS_ROOT = EXPERIMENT_ROOT / "analysis"
BASIS_SWEEP_ROOT = ANALYSIS_ROOT / "basis_sweep"
PRETRAIN_ROOT = EXPERIMENT_ROOT / "pretraining"
FIGURE_ROOT = REPO_ROOT / "notebooks" / "figures" / EXPERIMENT_NAME
FIGURE_LAYER_HEATMAP_NAME = "gpt2_tiny_layer_seed_space_error_heatmap"
FIGURE_ATTN_HEATMAP_NAME = "gpt2_tiny_attention_tensor_error_heatmap"
FIGURE_FF_HEATMAP_NAME = "gpt2_tiny_ff_tensor_error_heatmap"
FIGURE_HEATMAP_NAME = "gpt2_tiny_seed_space_tensor_error_heatmap"

CONFIG_PATH = (
    REPO_ROOT / "configs" / "gpt2_tiny_c4_ppt_reproduction.yaml"
)
config_payload = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
pre_cfg = config_payload["stages"]["prepretrain"]["training"]
TOKENS_PER_STEP = (
    int(pre_cfg["per_device_train_batch_size"])
    * int(pre_cfg.get("gradient_accumulation_steps", 1))
    * int(pre_cfg["block_size"])
)


def load_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    return json.loads(path.read_text(encoding="utf-8"))


def parse_degree(label: str) -> int | None:
    m = re.search(r"_d(\d+)$", label)
    return int(m.group(1)) if m else None


def init_display_name(_family: str, name: str) -> str:
    if name == "random":
        return "Random"
    if name == "weight_transfer":
        return "PPT"
    degree = parse_degree(name)
    if degree is not None:
        return f"Plato (d={degree})"
    return name.replace("_", " ").title()


def init_sort_key(name: str) -> tuple[int, int | str]:
    if name == "random":
        return (0, 0)
    if name == "weight_transfer":
        return (1, 0)
    degree = parse_degree(name)
    if degree is not None:
        return (2, degree)
    return (3, name)

def tensor_sort_key(name: str) -> tuple[int, int, int, str]:
    if name == "transformer.wte.weight":
        return (0, -1, 0, name)
    if name == "transformer.wpe.weight":
        return (1, -1, 0, name)

    layer_match = re.search(r"transformer\.h\.(\d+)\.(.+)", name)
    layer_idx = int(layer_match.group(1)) if layer_match else 10_000
    suffix = layer_match.group(2) if layer_match else name

    component_order = [
        "ln_1",
        "attn.c_attn",
        "attn.c_proj",
        "ln_2",
        "mlp.c_fc",
        "mlp.c_proj",
    ]
    component_rank = len(component_order)
    for idx, prefix in enumerate(component_order):
        if suffix.startswith(prefix):
            component_rank = idx
            break

    weight_bias_rank = 0 if suffix.endswith("weight") else 1 if suffix.endswith("bias") else 2
    return (2, layer_idx, component_rank * 10 + weight_bias_rank, name)


def tensor_layer(name: str) -> int | None:
    match = re.search(r"transformer\.h\.(\d+)\.", name)
    return int(match.group(1)) if match else None


def tensor_component_label(name: str) -> str:
    return name.replace("transformer.h.", "")


def is_attention_tensor(name: str) -> bool:
    return ".attn." in name


def is_ff_tensor(name: str) -> bool:
    return ".mlp." in name


## Load Experiment Summaries

This notebook reads only lightweight JSON summaries, so it can run locally without remote checkpoints.


In [2]:
weight_summary = load_json(ANALYSIS_ROOT / "weight_subspace_summary.json")
fit_manifest = load_json(BASIS_SWEEP_ROOT / "fit_blocks.json")
init_eval = load_json(PRETRAIN_ROOT / "init_eval.json", default={"results": []})
init_eval_curves = load_json(
    PRETRAIN_ROOT / "init_eval_basis_curves.json", default={"results": []}
)

fit_rows = []
for report_path in sorted(BASIS_SWEEP_ROOT.glob("analytic_fit_report_*.json")):
    slug = report_path.stem.replace("analytic_fit_report_", "")
    report = load_json(report_path)
    meta = fit_manifest.get(slug, {})
    fit_rows.append(
        {
            "fit_name": slug,
            "basis_type": meta.get("basis_type", "unknown"),
            "degree": parse_degree(slug),
            "mean_relative_error": report.get("mean_relative_error"),
            "num_tensors": len(report.get("tensors", {})),
            "report": report,
        }
    )
fit_df = pd.DataFrame(fit_rows)
if fit_df.empty:
    fit_df = pd.DataFrame(
        columns=[
            "fit_name",
            "basis_type",
            "degree",
            "mean_relative_error",
            "num_tensors",
            "report",
        ]
    )
else:
    fit_df = fit_df.sort_values(["degree", "fit_name"], na_position="last").reset_index(
        drop=True
    )
fit_name_order = fit_df.get("fit_name", pd.Series(dtype=str)).tolist()
fit_display_order = [init_display_name("plato", name) for name in fit_name_order]
if "fit_name" in fit_df:
    fit_df["fit_name"] = pd.Categorical(
        fit_df["fit_name"], categories=fit_name_order, ordered=True
    )
if "degree" in fit_df:
    fit_df["degree_label"] = fit_df["degree"].astype("Int64").astype(str)
else:
    fit_df["degree_label"] = pd.Series(dtype=str)
fit_df["display_name"] = pd.Categorical(
    fit_display_order, categories=fit_display_order, ordered=True
)

aes.set_palette(
    key="initializations",
    family="baseline",
    color="Greys",
    names=["random"],
    display_name=init_display_name,
)
aes.update_palette(
    key="initializations",
    family="transfer",
    color="Purples",
    names=["weight_transfer"],
    display_name=init_display_name,
)
if fit_name_order:
    aes.update_palette(
        key="initializations",
        family="plato",
        color="Oranges",
        names=list(reversed(fit_name_order)),
        display_name=init_display_name,
    )
INIT_PALETTE = dict(aes.PALETTES["initializations"])
INIT_LABEL_ORDER = ["Random", "PPT", *fit_display_order]
tensor_rows = []
for tensor_name, info in weight_summary.get("tensors", {}).items():
    evr = info.get("explained_variance_ratio", [])
    tensor_rows.append(
        {
            "tensor": tensor_name,
            "numel": info.get("numel"),
            "evr_1": evr[0] if len(evr) > 0 else None,
            "evr_2": evr[1] if len(evr) > 1 else None,
            "evr_cum_2": sum(evr[:2]) if evr else None,
        }
    )
seed_df = pd.DataFrame(tensor_rows)
if seed_df.empty:
    seed_df = pd.DataFrame(columns=["tensor", "numel", "evr_1", "evr_2", "evr_cum_2"])
else:
    seed_df = seed_df.sort_values("evr_cum_2", ascending=False).reset_index(drop=True)
seed_evr_map = (
    seed_df.set_index("tensor")["evr_cum_2"]
    if not seed_df.empty
    else pd.Series(dtype=float)
)
seed_numel_map = (
    seed_df.set_index("tensor")["numel"]
    if not seed_df.empty
    else pd.Series(dtype=float)
)

summary_results = init_eval.get("results", [])
curve_results = init_eval_curves.get("results") or summary_results

print(f"Experiment root: {EXPERIMENT_ROOT}")
print(f"Tokens per training step: {TOKENS_PER_STEP:,}")
summary_count = len(summary_results)
print(f"Found {len(fit_df)} fit reports and {summary_count} downstream run summaries.")
display(
    fit_df[
        ["display_name", "basis_type", "degree", "mean_relative_error", "num_tensors"]
    ]
)
display(seed_df.head(10))

Experiment root: /Users/jacksonpetty/Development/platonic-init/artifacts/experiments/gpt2_tiny_c4_ppt_reproduction
Tokens per training step: 8,192
Found 0 fit reports and 0 downstream run summaries.


,display_name,basis_type,degree,mean_relative_error,num_tensors


,tensor,numel,evr_1,evr_2,evr_cum_2


## Fit Blocks vs Original Pre-Pretraining Seeds

The analytic fit reports are computed on the PCA components extracted from the original pre-pretraining seeds.
Lower reconstruction error means a fit block better matches the shared seed-derived subspace.


In [3]:
seed_overview = pd.DataFrame(
    {
        "stat": [
            "num_tensors",
            "total_params_analyzed",
            "median_evr_cum_2",
            "mean_evr_cum_2",
        ],
        "value": [
            weight_summary.get("num_tensors"),
            weight_summary.get("total_params_analyzed"),
            seed_df["evr_cum_2"].median() if not seed_df.empty else None,
            seed_df["evr_cum_2"].mean() if not seed_df.empty else None,
        ],
    }
)
display(seed_overview)
display(fit_df[["display_name", "degree", "mean_relative_error", "num_tensors"]])

,stat,value
0,num_tensors,None
1,total_params_analyzed,None
2,median_evr_cum_2,None
3,mean_evr_cum_2,None


,display_name,degree,mean_relative_error,num_tensors


In [4]:
if fit_df.empty:
    print("No analytic fit reports found yet; skipping fit-error bar plot.")
else:
    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN * 1.9))

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.barplot(
            data=fit_df,
            x="degree_label",
            y="mean_relative_error",
            hue="display_name",
            hue_order=fit_display_order,
            palette=INIT_PALETTE,
            legend=False,
            ax=ax,
        )
        ax.set_title("Analytic Fit Error vs Seed-Derived Subspace")
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Mean component relative error")
        ax.tick_params(axis="x", rotation=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    aes.save_figure(FIGURE_ROOT / "gpt2_tiny_fit_error_vs_degree", fig=fig)
    plt.show()


No analytic fit reports found yet; skipping fit-error bar plot.


**Note**
The token/position embedding tensors (`wte`/`wpe`) are excluded from the heatmap below. They are persistently hard to fit, but that is less informative here because embeddings are reset/reprojected before downstream pretraining.


In [5]:
if fit_df.empty or seed_df.empty:
    print("Fit reports or seed-summary statistics are not available yet; skipping fit heatmaps.")
else:
    tensor_error_rows = []
    for _, row in fit_df.iterrows():
        for tensor_name, info in row["report"].get("tensors", {}).items():
            tensor_error_rows.append(
                {
                    "fit_name": row["fit_name"],
                    "degree": row["degree"],
                    "tensor": tensor_name,
                    "layer": tensor_layer(tensor_name),
                    "component": tensor_component_label(tensor_name),
                    "mean_component_relative_error": info.get(
                        "mean_component_relative_error"
                    ),
                    "numel": seed_numel_map.get(tensor_name),
                }
            )

    tensor_error_df = pd.DataFrame(tensor_error_rows)
    non_embedding_tensor_error_df = tensor_error_df[
        ~tensor_error_df["tensor"].str.contains(r"wte|wpe", regex=True)
    ].copy()
    degree_lookup = fit_df.set_index("fit_name")["degree"].to_dict()

    layer_error_df = (
        non_embedding_tensor_error_df.dropna(subset=["layer", "numel"])
        .groupby(["layer", "fit_name"], as_index=False)
        .apply(
            lambda frame: pd.Series(
                {
                    "weighted_error": np.average(
                        frame["mean_component_relative_error"],
                        weights=frame["numel"],
                    )
                }
            ),
            include_groups=False,
        )
    )
    layer_heatmap_df = (
        layer_error_df.pivot(index="layer", columns="fit_name", values="weighted_error")
        .reindex(columns=fit_name_order)
        .sort_index(ascending=False)
    )
    layer_heatmap_df.index = [f"layer {int(layer)}" for layer in layer_heatmap_df.index]
    layer_heatmap_df.columns = [str(int(degree_lookup[name])) for name in layer_heatmap_df.columns]

    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, max(2.4, 0.45 * len(layer_heatmap_df))))
    with sns.plotting_context("paper", font_scale=0.9, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.heatmap(layer_heatmap_df, cmap="magma_r", vmin=0.0, vmax=1.0, ax=ax)
        ax.set_title("Layer-Aggregated Seed-Space Error")
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Layer")
        ax.tick_params(axis="x", rotation=0)

    aes.save_figure(FIGURE_ROOT / FIGURE_LAYER_HEATMAP_NAME, fig=fig)
    plt.show()


    def component_heatmap_df(filter_fn) -> pd.DataFrame:
        component_df = non_embedding_tensor_error_df[
            non_embedding_tensor_error_df["tensor"].map(filter_fn)
        ].copy()
        component_df = component_df.dropna(subset=["layer"])
        component_df["row_label"] = component_df.apply(
            lambda row: f"layer {int(row['layer'])}: {row['component']}", axis=1
        )
        heatmap_df = (
            component_df.pivot(
                index="row_label", columns="fit_name", values="mean_component_relative_error"
            )
            .reindex(columns=fit_name_order)
        )
        ordered_rows = (
            component_df[["row_label", "layer", "tensor"]]
            .drop_duplicates()
            .assign(sort_key=lambda frame: frame["tensor"].map(tensor_sort_key))
            .sort_values("sort_key", ascending=False)["row_label"]
            .tolist()
        )
        heatmap_df = heatmap_df.reindex(ordered_rows)
        heatmap_df.columns = [str(int(degree_lookup[name])) for name in heatmap_df.columns]
        return heatmap_df


    attn_heatmap_df = component_heatmap_df(is_attention_tensor)
    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, max(2.8, 0.24 * len(attn_heatmap_df))))
    with sns.plotting_context("paper", font_scale=0.8, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.heatmap(attn_heatmap_df, cmap="magma_r", vmin=0.0, vmax=1.0, ax=ax)
        ax.set_title("Attention Tensor Error by Layer")
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Tensor")
        ax.tick_params(axis="x", rotation=0)

    aes.save_figure(FIGURE_ROOT / FIGURE_ATTN_HEATMAP_NAME, fig=fig)
    plt.show()

    ff_heatmap_df = component_heatmap_df(is_ff_tensor)
    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, max(2.8, 0.24 * len(ff_heatmap_df))))
    with sns.plotting_context("paper", font_scale=0.8, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.heatmap(ff_heatmap_df, cmap="magma_r", vmin=0.0, vmax=1.0, ax=ax)
        ax.set_title("Feed-Forward Tensor Error by Layer")
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Tensor")
        ax.tick_params(axis="x", rotation=0)

    aes.save_figure(FIGURE_ROOT / FIGURE_FF_HEATMAP_NAME, fig=fig)
    plt.show()


Fit reports or seed-summary statistics are not available yet; skipping fit heatmaps.


## Downstream Pretraining Loss Curves by Initialization

This section plots the available downstream eval-loss curves from `pretraining/init_eval*.json`,
with one series per initialization label and training tokens on the x-axis.


In [6]:
result_rows = []
for row in summary_results:
    raw_label = row.get("label", row.get("variant", "unknown"))
    result_rows.append(
        {
            "label": init_display_name("", raw_label),
            "raw_label": raw_label,
            "variant": row.get("variant"),
            "init_mode": row.get("init_mode"),
            "best_eval_loss": row.get("best_eval_loss"),
            "final_eval_loss": row.get("final_eval_loss"),
            "train_loss": row.get("train_loss"),
        }
    )
result_df = pd.DataFrame(result_rows)
if result_df.empty:
    result_df = pd.DataFrame(
        columns=[
            "label",
            "raw_label",
            "variant",
            "init_mode",
            "best_eval_loss",
            "final_eval_loss",
            "train_loss",
        ]
    )
else:
    result_df = result_df.sort_values("best_eval_loss").reset_index(drop=True)
display(result_df)

curve_rows = []
for row in curve_results:
    raw_label = row.get("label", row.get("variant", "unknown"))
    display_label = init_display_name("", raw_label)
    for point in row.get("eval_curve", []):
        step = point.get("step")
        curve_rows.append(
            {
                "label": display_label,
                "raw_label": raw_label,
                "step": step,
                "tokens": step * TOKENS_PER_STEP if step is not None else None,
                "loss": point.get("eval_loss"),
                "metric": "eval_loss",
                "basis": row.get("basis"),
                "init_mode": row.get("init_mode"),
            }
        )
    for point in row.get("train_curve") or []:
        step = point.get("step")
        curve_rows.append(
            {
                "label": display_label,
                "raw_label": raw_label,
                "step": step,
                "tokens": step * TOKENS_PER_STEP if step is not None else None,
                "loss": point.get("loss"),
                "metric": "train_loss",
                "basis": row.get("basis"),
                "init_mode": row.get("init_mode"),
            }
        )

curves_df = pd.DataFrame(curve_rows)
available_labels = (
    sorted(curves_df.label.unique().tolist()) if not curves_df.empty else []
)
print(f"Available run labels: {available_labels}")
available_metrics = (
    sorted(curves_df.metric.unique().tolist()) if not curves_df.empty else []
)
print(f"Available metrics: {available_metrics}")
display(curves_df.head())


,label,raw_label,variant,init_mode,best_eval_loss,final_eval_loss,train_loss


Available run labels: []
Available metrics: []


""


In [7]:
if curves_df.empty:
    print("No downstream pretraining curves were found in the pulled summaries yet; skipping loss plot.")
    plot_df = pd.DataFrame()
    label_order = []
else:
    metric_to_plot = "eval_loss"
    selected_labels = ["Random", "PPT", "Plato (d=6)", "Plato (d=24)"]
    plot_df = curves_df[curves_df["metric"] == metric_to_plot].copy()
    plot_df = plot_df[plot_df["label"].isin(selected_labels)].copy()
    label_order = [label for label in selected_labels if label in plot_df["label"].unique()]

    random_best_loss = None
    random_best_tokens = None
    random_rows = plot_df[plot_df["label"] == "Random"]
    if not random_rows.empty:
        best_random = random_rows.sort_values(["loss", "tokens"]).iloc[0]
        random_best_loss = float(best_random["loss"])
        random_best_tokens = float(best_random["tokens"])

    threshold_markers = []
    legend_label_map = {label: label for label in label_order}
    if (
        random_best_loss is not None
        and random_best_tokens is not None
        and random_best_tokens > 0
    ):
        for label in label_order:
            if label == "Random":
                continue
            label_rows = plot_df[plot_df["label"] == label].sort_values("tokens")
            crossings = label_rows[label_rows["loss"] <= random_best_loss]
            if crossings.empty:
                final_loss = float(label_rows.iloc[-1]["loss"])
                gap = final_loss - random_best_loss
                legend_label_map[label] = f"{label} • +{gap:.3f} nats/token"
                continue
            first_crossing = crossings.iloc[0]
            crossing_tokens = float(first_crossing["tokens"])
            crossing_loss = float(random_best_loss)
            crossing_pos = label_rows.index.get_loc(first_crossing.name)
            if crossing_pos > 0:
                prev_row = label_rows.iloc[crossing_pos - 1]
                prev_tokens = float(prev_row["tokens"])
                prev_loss = float(prev_row["loss"])
                next_tokens = float(first_crossing["tokens"])
                next_loss = float(first_crossing["loss"])
                if prev_loss > random_best_loss and next_loss < prev_loss:
                    loss_delta = next_loss - prev_loss
                    if loss_delta != 0:
                        frac = (random_best_loss - prev_loss) / loss_delta
                        crossing_tokens = prev_tokens + frac * (next_tokens - prev_tokens)
            savings_pct = 100.0 * (1.0 - crossing_tokens / random_best_tokens)
            legend_label_map[label] = f"{label} • {savings_pct:.0f}% savings"
            threshold_markers.append(
                {
                    "label": label,
                    "tokens": crossing_tokens,
                    "loss": crossing_loss,
                }
            )

    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN))
    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.lineplot(
            data=plot_df,
            x="tokens",
            y="loss",
            hue="label",
            hue_order=label_order,
            palette=INIT_PALETTE,
            linewidth=2.0,
            ax=ax,
        )
        if threshold_markers:
            marker_df = pd.DataFrame(threshold_markers)
            sns.scatterplot(
                data=marker_df,
                x="tokens",
                y="loss",
                hue="label",
                hue_order=label_order,
                palette=INIT_PALETTE,
                s=80,
                legend=False,
                ax=ax,
                zorder=5,
            )
        if random_best_loss is not None:
            ax.axhline(
                random_best_loss,
                color=aes.darken("#999999", by=0.15),
                linewidth=1.0,
                linestyle="--",
            )
        ax.set_title("Loss by Initialization (GPT-2 Tiny)")
        ax.set_xlabel("Training tokens")
        ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
        ax.set_ylabel("Validation loss (C4)")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        legend = ax.legend(loc="upper right", frameon=False, title=None)
        if legend is not None:
            for text in legend.texts:
                raw = text.get_text()
                text.set_text(legend_label_map.get(raw, raw))

    aes.save_figure(
        FIGURE_ROOT / "gpt2_tiny_downstream_eval_loss_selected_initializations",
        fig=fig,
    )
    plt.show()


No downstream pretraining curves were found in the pulled summaries yet; skipping loss plot.


## Token Efficiency vs Random

This plot compares each initialization against the random baseline at matched validation loss.
For a run point at token budget `t`, it finds the earliest point where the random initialization reaches the same or lower eval loss, then reports the percent of tokens saved:

`100 * (1 - t / t_random_match)`

Positive values mean the initialization reaches the same loss with fewer tokens than random.


In [8]:
if plot_df.empty:
    print("No downstream loss curves available yet; skipping token-efficiency plot.")
else:
    efficiency_df = plot_df.copy()
    random_df = (
        plot_df[plot_df["label"] == "Random"][["tokens", "loss"]]
        .dropna()
        .sort_values("tokens")
        .reset_index(drop=True)
    )

    if random_df.empty:
        print("Random baseline curve is required for efficiency analysis; skipping.")
    else:
        random_df = random_df[random_df["tokens"] > 0].reset_index(drop=True)
        random_df["best_loss"] = random_df["loss"].cummin()
        random_tokens = random_df["tokens"].to_numpy(dtype=float)
        random_best_loss = random_df["best_loss"].to_numpy(dtype=float)

        def random_tokens_to_reach_loss(target_loss: float) -> float | None:
            match_idx = np.flatnonzero(random_best_loss <= target_loss)
            if len(match_idx) == 0:
                return None
            return float(random_tokens[match_idx[0]])

        efficiency_rows = []
        for row in efficiency_df.itertuples(index=False):
            tokens = float(row.tokens)
            if tokens <= 0:
                continue
            if row.label == "Random":
                random_match_tokens = tokens
                tokens_saved_pct = 0.0
            else:
                random_match_tokens = random_tokens_to_reach_loss(float(row.loss))
                if random_match_tokens is None or random_match_tokens <= 0:
                    continue
                tokens_saved_pct = 100.0 * (1.0 - tokens / random_match_tokens)
            efficiency_rows.append(
                {
                    "label": row.label,
                    "tokens": tokens,
                    "loss": float(row.loss),
                    "random_match_tokens": random_match_tokens,
                    "tokens_saved_pct": tokens_saved_pct,
                }
            )

        efficiency_plot_df = pd.DataFrame(efficiency_rows)
        if efficiency_plot_df.empty:
            print("No comparable efficiency points found yet; skipping token-efficiency plot.")
        else:
            fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN))
            with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
                ax = fig.add_subplot(1, 1, 1)
                sns.lineplot(
                    data=efficiency_plot_df,
                    x="tokens",
                    y="tokens_saved_pct",
                    hue="label",
                    hue_order=label_order,
                    palette=INIT_PALETTE,
                    linewidth=2.0,
                    ax=ax,
                )
                ax.axhline(
                    0.0,
                    color=aes.darken("#999999", by=0.15),
                    linewidth=1.0,
                    linestyle="--",
                )
                ax.set_title("Token Efficiency vs Random (GPT-2 Tiny)")
                ax.set_xlabel("Training tokens")
                ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
                ax.set_ylabel("Tokens saved vs Random (%)")
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)

            aes.save_figure(
                FIGURE_ROOT / "gpt2_tiny_token_efficiency_vs_random",
                fig=fig,
            )
            plt.show()


No downstream loss curves available yet; skipping token-efficiency plot.


## Notes

- The fit-quality section is tied directly to the original pre-pretraining seeds because the analytic fit reports are computed on the seed-derived PCA subspace.
- The downstream section plots whatever run summaries are currently present on disk. If only `random` and `weight_transfer` are available so far, the notebook will reflect that.
- Once fit-based downstream runs finish and their summaries are synced, rerunning this notebook should extend the curve plot automatically.
